In [0]:
###Imports

In [0]:
###Conexões

In [0]:
query = """
(
    SELECT 
        op.id,
        case 
            when op.id_enum_type_solicitation = 43 then op_mae.txt_policy_number
            else op.txt_policy_number
        end AS txt_policy_number,
        case 
            when op.id_enum_type_solicitation = 43 then op.txt_policy_number
            else null
        end as txt_endosso,
        cic.txt_cnpj as cnpj_seguradora,
        op.txt_documento_base,
        op.id_insurance_company
    FROM operational_policy op
    LEFT JOIN core_insurance_company cic 
        ON op.id_insurance_company = cic.id
    LEFT JOIN operational_policy op_mae
        ON op.id_initial_policy = op_mae.id
    WHERE op.source = 0 
        and op.dt_emission > '2025-01-01'
        and
        (
            op.vl_premium > 4000
            OR op.vl_insured_value > 1000000
        )
        AND op.dt_logical_delete IS NULL
        AND op.dt_start_policy IS NOT NULL
        AND op.dt_end_policy IS NOT NULL
        AND op.enum_endorsement_type <> 1
        AND NOW() BETWEEN op.dt_start_policy AND op.dt_end_policy
        AND
        (
            op.status IN (6, 10, 14, 18)
            OR
            (
                op.status = 17
                AND EXISTS (
                    SELECT 1
                    FROM operational_policy_renew opr
                    WHERE opr.id_policy_old = op.id
                      AND opr.status = 4
                )
            )
        )
) query_df


"""



In [0]:
df = spark.read.jdbc(
    url=jdbcUrl,
    table=query,
    properties=connectionProperties
)

df.createOrReplaceTempView("fato_operacional")


In [0]:
bucket_carlos = """


select * from fato_operacional
where id_insurance_company not in (1,2, 21)
and txt_documento_base is not null

"""

spark.sql(bucket_carlos).createOrReplaceTempView("vw_bucket_carlos")


In [0]:
df_controle = spark.table("hive_metastore.default.policy_document_read_control")
df_controle.createOrReplaceTempView("view_control")



In [0]:
### Microsoft Conexões

In [0]:
#token

In [0]:
###URL_PA

In [0]:
#token

In [0]:
df_controle = spark.table("hive_metastore.default.policy_document_read_control")
df_controle.createOrReplaceTempView("view_control")

In [0]:
lote = """
    SELECT policy_id
    FROM view_control
    WHERE status = 'PENDENTE'
    order by 1
    LIMIT 10
"""
lote_df = spark.sql(lote)
lote_df.createOrReplaceTempView("lote")

if lote_df.count() == 0:
    print("Nada pendente")
    dbutils.notebook.exit("OK")




In [0]:
spark.sql("""
MERGE INTO policy_document_read_control t
USING lote l
ON t.policy_id = l.policy_id
WHEN MATCHED AND t.status = 'PENDENTE' THEN
  UPDATE SET
    t.status = 'EM_PROCESSAMENTO',
    t.dt_primeiro_envio = CASE
        WHEN t.dt_primeiro_envio IS NULL THEN current_timestamp()
        ELSE t.dt_primeiro_envio
    END,
    t.dt_ultima_tentativa = CASE
        WHEN t.dt_primeiro_envio IS NOT NULL THEN current_timestamp()
        ELSE t.dt_ultima_tentativa
    END

""").show


In [0]:
df_controle = spark.table("hive_metastore.default.policy_document_read_control")
df_controle.createOrReplaceTempView("view_control")

In [0]:
hashes_df = spark.sql("""
    SELECT 
        vc.document_hash
    FROM view_control vc
    WHERE vc.status = 'EM_PROCESSAMENTO'
    ORDER BY vc.document_hash
    LIMIT 10
""")


hashes = [f"'{row.document_hash}'" for row in hashes_df.collect()]
hashes_str = ",".join(hashes)

df_si = spark.read.jdbc(
    url=jdbcUrl,
    table=f"""
    (
        SELECT
            txt_name,
            content,
            SUBSTRING(
            txt_extension,
            LOCATE('data:', txt_extension) + 5,
            LOCATE(';base64', txt_extension) - (LOCATE('data:', txt_extension) + 5))             as txt_extension
        FROM storage_image
        WHERE txt_name IN ({hashes_str})
    ) tmp
    """,
    properties=connectionProperties
)



In [0]:
def enviar_para_ia(txt_name, content, txt_extension):
    body = {
        "nome_arquivo": txt_name,
        "conteudo_arquivo": content,
        "tipo_conteudo": txt_extension
    }

    try:
        response = requests.post(
            url_pa,
            headers=headers_pa,
            json=body,
            timeout=60
        )

        return response.status_code, response.json()

    except Exception as e:
        return "EXCEPTION", str(e)


In [0]:
for row in df_si.toLocalIterator():
    status_code, payload = enviar_para_ia(
        row.txt_name,
        row.content,
        row.txt_extension
    )

    payload_str = str(payload).replace("'", "''")
    dados_leitura = payload.get("dados_leitura") if payload else None

    status = "ERRO_DESCONHECIDO"

    if status_code == 200 and dados_leitura:
        status = "PROCESSADO"

        valor_is = dados_leitura.get("valor_is")
        valor_premio = dados_leitura.get("valor_premio")
        cnpj_seguradora = dados_leitura.get("cnpj_seguradora")
        inicio_vig = dados_leitura.get("data_inicio_vigencia")
        fim_vig = dados_leitura.get("data_fim_vigencia")
        dt_emissao = dados_leitura.get("data_emissao")

        spark.sql(f"""
        UPDATE policy_document_read_control
        SET
            status = '{status}',
            valor_is = {valor_is if valor_is is not None else 'NULL'},
            valor_premio = {valor_premio if valor_premio is not None else 'NULL'},
            inicio_vigencia = {f"timestamp('{inicio_vig}')" if inicio_vig else 'NULL'},
            fim_vigencia = {f"timestamp('{fim_vig}')" if fim_vig else 'NULL'},
            dt_emissiao = {f"timestamp('{dt_emissao}')" if dt_emissao else 'NULL'},
            reader_response_json = '{payload_str}',
            dt_ultima_tentativa = current_timestamp(),
            tentativas = coalesce(tentativas, 0) + 1
        WHERE document_hash = '{row.txt_name}'
        """)

    elif status_code == 200 and not dados_leitura:
        status = "SUCESSO_SEM_DADOS"

        spark.sql(f"""
        UPDATE policy_document_read_control
        SET
            status = '{status}',
            reader_response_json = '{payload_str}',
            dt_ultima_tentativa = current_timestamp(),
            tentativas = coalesce(tentativas, 0) + 1
        WHERE document_hash = '{row.txt_name}'
        """)

    elif status_code in (500, 502):
        status = "ERRO_TEMP"

        spark.sql(f"""
        UPDATE policy_document_read_control
        SET
            status = '{status}',
            mensagem_erro = '{payload_str}',
            dt_ultima_tentativa = current_timestamp(),
            tentativas = coalesce(tentativas, 0) + 1
        WHERE document_hash = '{row.txt_name}'
        """)

    elif status_code in (400, 422):
        status = "DOC_INVALIDO"

        spark.sql(f"""
        UPDATE policy_document_read_control
        SET
            status = '{status}',
            mensagem_erro = '{payload_str}',
            dt_ultima_tentativa = current_timestamp(),
            tentativas = coalesce(tentativas, 0) + 1
        WHERE document_hash = '{row.txt_name}'
        """)

    else:
        status = "ERRO_DESCONHECIDO"

        spark.sql(f"""
        UPDATE policy_document_read_control
        SET
            status = '{status}',
            mensagem_erro = '{payload_str}',
            dt_ultima_tentativa = current_timestamp(),
            tentativas = coalesce(tentativas, 0) + 1
        WHERE document_hash = '{row.txt_name}'
        """)
